# Assignment 2B — RAG Pipeline on *The Constitution of India (2026)*

This notebook builds a small, complete retrieval-augmented generation (RAG) system on the Assignment 1 corpus:

```text
corpus_clean.txt ──fixed-size chunks──► dense (bge + FAISS) │ BM25 │ hybrid (RRF) ──► cross-encoder rerank ──► top-3 chunks
                                                                                                             │
           LLM judge (correct? supported?) ◄── grounded answer with [n] citations ◄── Assignment 1 model ◄──┘
```

### Group Members

| Sr. No. | Student ID | Name |
| :---: | :--- | :--- |
| 1 | `2025ae05573` | Darshil Sanjay Shah |
| 2 | `2025ae05830` | Prashant V Panchal |
| 3 | `2025ae05839` | Jena Ajit Anirudha Ranjita |
| 4 | `2025ae05840` | Kulkarni Divya Prakash Revati |

**Reused from Assignment 1:**
* the cleaned corpus `corpus_clean.txt`, which holds 518 sections (Preamble, articles, schedules, appendices);
* 10 questions from the held-out evaluation split of `instructions.jsonl`, with their expected answers;
* the fine-tuned model (Qwen3-0.6B-Base → CPT → SFT), retrained by `a1_retrain_model.ipynb` and stored on Google Drive.

### Before you run (Colab)
1. **Runtime → Change runtime type → T4 GPU**.
2. Run `a1_retrain_model.ipynb` first. It puts the model under `MyDrive/llm_assignment_2/models/`.
3. `MyDrive/llm_assignment_2/data/` must contain `sections.jsonl`, `instructions.jsonl` and `corpus_clean.txt`.
4. *(Optional)* Add a Colab secret **`HF_TOKEN`** (🔑 icon in the left bar). It is only needed if your Hugging Face account has access to the gated `meta-llama/Llama-3.1-8B-Instruct` judge. Without it, the notebook picks an open judge model automatically.

In [27]:
# ── Paths and run mode ──
import os, sys, json, re, time, gc, math, bisect, subprocess, collections
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
SMOKE_TEST = os.environ.get("SMOKE_TEST") == "1"    # CPU test run with tiny LLMs (Parts A–C are unchanged)

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = Path("/content/drive/MyDrive/llm_assignment_2")
    try:
        from google.colab import userdata
        os.environ.setdefault("HF_TOKEN", userdata.get("HF_TOKEN") or "")
    except Exception:
        pass                                        # no secret set: open models only
else:
    PROJECT_DIR = Path(os.environ.get("PROJECT_DIR", ".")).resolve()

DATA_DIR = PROJECT_DIR / "data"
MODELS_DIR = PROJECT_DIR / ("models_smoke" if SMOKE_TEST else "models")
RESULTS_DIR = PROJECT_DIR / "results" / "rag"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
HF_TOKEN = os.environ.get("HF_TOKEN") or None

for f in ["sections.jsonl", "instructions.jsonl", "corpus_clean.txt"]:
    assert (DATA_DIR / f).exists(), f"missing {DATA_DIR / f}: upload it from llm_assingment_1/"
assert (MODELS_DIR / "a1_model_info.json").exists(), f"run a1_retrain_model.ipynb first (no model in {MODELS_DIR})"
print("project:", PROJECT_DIR, "| smoke test:", SMOKE_TEST, "| HF token:", "yes" if HF_TOKEN else "no")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
project: /content/drive/MyDrive/llm_assignment_2 | smoke test: False | HF token: no


In [28]:
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "faiss-cpu", "rank_bm25",
                    "sentence-transformers", "bitsandbytes>=0.46.1", "peft"], check=True)

In [29]:
import numpy as np
import pandas as pd
import torch
import faiss
import transformers
from IPython.display import display, Markdown
from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, set_seed
from peft import PeftModel

set_seed(0)
pd.set_option("display.max_colwidth", None)        # show answers and judge reasons in full
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
HAS_GPU = DEVICE == "cuda"
COMPUTE_DTYPE = (torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16) if HAS_GPU else torch.float32
from packaging import version
DTYPE_KW = "dtype" if version.parse(transformers.__version__) >= version.parse("4.56") else "torch_dtype"
print(f"device {DEVICE} ({torch.cuda.get_device_name(0) if HAS_GPU else 'CPU'}) | torch {torch.__version__} | "
      f"transformers {transformers.__version__} | faiss {faiss.__version__}")

def free_memory():
    gc.collect()
    if HAS_GPU:
        torch.cuda.empty_cache()

def check_quoted(actual, quoted, where):
    """The analysis text quotes numbers from our run; warn if this run produced different ones."""
    diff = {k: f"text says {quoted[k]}, this run {actual[k]:g}" for k in quoted if abs(actual[k] - quoted[k]) > 1e-6}
    print(f"✓ the numbers quoted in {where} match this run" if not diff else
          f"⚠ {where}: results differ from the numbers quoted in the text, please update it → {diff}")

def sync():
    """Wait for queued GPU work so wall-clock timings are honest."""
    if HAS_GPU:
        torch.cuda.synchronize()

device cuda (Tesla T4) | torch 2.14.1+cu130 | transformers 5.22.1 | faiss 1.15.1


## Data: corpus, 10 evaluation queries and their answer spans

The corpus is `corpus_clean.txt` from Assignment 1. It is the 518 sections joined together, each with a provenance header such as `The Constitution of India — Part V …\nArticle 85: Sessions of Parliament…`. We also rebuild where each section starts in the text, so that every chunk can be labelled with the article(s) it comes from.

**The 10 queries** come from the held-out evaluation split of Assignment 1. They cover 6 factual questions, 1 procedural, 1 explanatory, 1 scenario, and 1 question whose wording differs from the legal text ("Lok Sabha / Rajya Sabha" in the query against "House of the People / Council of States" in the corpus).

**Answer spans.** For each query we marked the shortest sentence or clause in the corpus that answers it, copied word for word. A retrieved chunk is a **hit** if it contains that span, after collapsing whitespace. The cell below checks that every span occurs **exactly once** in the corpus. One example of why this matters: Article 174 (State Legislatures) repeats the "six months shall not intervene" wording of Article 85, so the span for that query includes "each House of Parliament" to rule out the State provision.

In [30]:
corpus = (DATA_DIR / "corpus_clean.txt").read_text()
sections = [json.loads(l) for l in open(DATA_DIR / "sections.jsonl")]
pairs = {p["id"]: p for p in map(json.loads, open(DATA_DIR / "instructions.jsonl"))}

def as_document(s):   # identical to Assignment 1
    return f"The Constitution of India — {s['part']}\n{s['id']}: {s['title']}\n\n{s['text']}" \
        if s["kind"] == "article" else f"The Constitution of India — {s['part']}\n\n{s['text']}"

docs = [as_document(s) for s in sections]
assert "\n\n".join(docs) == corpus
section_starts, pos = [], 0
for d in docs:
    section_starts.append(pos); pos += len(d) + 2

def norm(t):
    return re.sub(r"\s+", " ", t).strip()

ANSWER_SPANS = {   # query id → answer span (verbatim from the corpus)
    "coi-011": "The State shall not discriminate against any citizen on grounds only of religion, race, caste, sex, place of birth or any of them",
    "coi-024": "No child below the age of fourteen years shall be employed to work in any factory or mine",
    "coi-058": "election of a President or Vice-President shall be inquired into and decided by the Supreme Court",
    "coi-067": "not more than five hundred and thirty members chosen by direct election from territorial constituencies in the States",
    "coi-070": "summon each House of Parliament to meet at such time and place as he thinks fit, but six months shall not intervene",
    "coi-094": "A Judge of the Supreme Court shall not be removed from his office except by an order of the President",
    "coi-129": "seats to be filled by direct election in every Panchayat shall be reserved for women",
    "coi-138": "the maximum number of directors of a co-operative society shall not exceed twenty-one",
    "coi-221": "not less than two-thirds of the members of the legislature party concerned have agreed to such merger",
    "coi-248": "in the case of a seat in the Council of States, not less than thirty years of age and, in the case of a seat in the House of the People, not less than twenty-five years of age",
}
corpus_norm = norm(corpus)
queries = []
for qid, span in ANSWER_SPANS.items():
    p = pairs[qid]
    assert p["split"] == "eval", qid
    n = corpus_norm.count(norm(span))
    assert n == 1, f"{qid}: span occurs {n} times in the corpus"
    queries.append(dict(id=qid, question=p["instruction"], expected=p["response"], span=span,
                        source=p["source_section"], type=p["question_type"]))
Q = [q["question"] for q in queries]
display(pd.DataFrame(queries)[["id", "type", "source", "question", "span", "expected"]])

,id,type,source,question,span,expected
0,coi-011,factual,Article 15,On which grounds does the Constitution forbid the State from discriminating against citizens under Article 15?,"The State shall not discriminate against any citizen on grounds only of religion, race, caste, sex, place of birth or any of them","Article 15 bars the State from discriminating against any citizen only on grounds of religion, race, caste, sex or place of birth. It also forbids restrictions on access to shops, restaurants, wells and public places on those grounds, while still allowing special provisions for women, children and backward classes."
1,coi-024,factual,Article 24,What does the Constitution say about employing children in factories?,No child below the age of fourteen years shall be employed to work in any factory or mine,"Article 24 prohibits employing any child below fourteen years of age in a factory, mine or any other hazardous employment. It complements the directive principles that call on the State to protect children from abuse and from being forced into work unsuited to their age."
2,coi-058,factual,Article 71,Which court decides disputes about the election of the President or Vice-President?,election of a President or Vice-President shall be inquired into and decided by the Supreme Court,"Article 71 provides that all doubts and disputes arising from the election of a President or Vice-President are inquired into and decided exclusively by the Supreme Court, whose decision is final. Acts done by that person before the election is set aside remain valid."
3,coi-067,factual,Article 81,What is the maximum strength of the House of the People?,not more than five hundred and thirty members chosen by direct election from territorial constituencies in the States,"Article 81 caps the House of the People at not more than five hundred and thirty members directly elected from the States and not more than twenty members representing the Union territories. Seats are allotted so that the population-to-seat ratio is, as far as practicable, the same for all States."
4,coi-070,factual,Article 85,What is the maximum permitted gap between two sessions of Parliament?,"summon each House of Parliament to meet at such time and place as he thinks fit, but six months shall not intervene",Article 85 requires the President to summon each House so that six months do not pass between the last sitting of one session and the first sitting of the next. The President may also prorogue either House and dissolve the House of the People.
5,coi-094,procedural,Article 124,How is a judge of the Supreme Court removed?,A Judge of the Supreme Court shall not be removed from his office except by an order of the President,"Under Article 124(4), a judge can be removed only by a presidential order after each House of Parliament passes an address, in the same session, supported by a majority of its total membership and two-thirds of members present and voting. The only grounds are proved misbehaviour or incapacity."
6,coi-129,factual,Article 243D,What share of Panchayat seats is reserved for women?,seats to be filled by direct election in every Panchayat shall be reserved for women,"Article 243D reserves at least one-third of all directly elected Panchayat seats for women, counting the women's share within seats reserved for Scheduled Castes and Scheduled Tribes. These seats rotate among constituencies, and the State Legislature must also reserve chairperson posts for women in the manner it decides by law."
7,coi-138,factual,Article 243ZJ,What is the maximum size of the board of a co-operative society?,the maximum number of directors of a co-operative society shall not exceed twenty-one,"Article 243ZJ caps a co-operative society's board at twenty-one directors, with one seat reserved for Scheduled Castes or Scheduled Tribes and two for women. Elected members and office bearers serve five-year terms."
8,coi-221,explanatory,Tenth Schedule,Is a merger of political parties treated as

## Part A — Dense search and chunk size

**Embedding model and its input limit.** The brief's example model, `all-MiniLM-L6-v2`, truncates input at **256 tokens**. Two of the chunk sizes we test, 400 and 500, are larger than that, so the end of every such chunk would be silently cut off when embedded. As the brief allows, **we switched to a model with a longer limit**: **`BAAI/bge-base-en-v1.5`**, which takes **512 tokens** and produces 768-dimensional vectors. 500 tokens plus `[CLS]`/`[SEP]` is 502, which fits. Token counts below use bge's own tokenizer (BERT uncased WordPiece). bge expects a short instruction in front of each *query*, though not in front of passages: `"Represent this sentence for searching relevant passages: "`.

The brief's chunk-size table lists 100 and 500 tokens, while its text and checklist say 100 and 400. **We run all three: 100, 400 and 500.**

In [31]:
EMBED_MODEL = "BAAI/bge-base-en-v1.5"
QUERY_PREFIX = "Represent this sentence for searching relevant passages: "
EMBED_BATCH = 64

embedder = SentenceTransformer(EMBED_MODEL, device=DEVICE)
emb_tok = embedder.tokenizer
minilm_tok = AutoTokenizer.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")
print(f"{EMBED_MODEL}: max input {embedder.max_seq_length} tokens, "
      f"dim {embedder.get_sentence_embedding_dimension()} | all-MiniLM-L6-v2 limit: 256 tokens")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BAAI/bge-base-en-v1.5: max input 512 tokens, dim 768 | all-MiniLM-L6-v2 limit: 256 tokens


/tmp/ipykernel_2840/2120189390.py:9: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  f"dim {embedder.get_sentence_embedding_dimension()} | all-MiniLM-L6-v2 limit: 256 tokens")


### A1.1 — One reusable dense-search function

`build_dense_search(chunks)` embeds the chunks with L2-normalised vectors and adds them to a FAISS **`IndexFlatIP`**, an exact inner-product index. Because the vectors are normalised, inner product equals cosine similarity. The function returns a `search(query, k)` function, which gives the top-k `(chunk_id, score)` pairs, together with build statistics.

In [32]:
def build_dense_search(chunks, model=embedder, batch_size=EMBED_BATCH, query_prefix=QUERY_PREFIX):
    """Embed `chunks` (list of str), build a FAISS IndexFlatIP, return (search_fn, stats)."""
    sync(); t0 = time.perf_counter()
    vecs = model.encode(chunks, batch_size=batch_size, normalize_embeddings=True,
                        convert_to_numpy=True, show_progress_bar=False).astype("float32")
    index = faiss.IndexFlatIP(vecs.shape[1])
    index.add(vecs)
    sync(); build_s = time.perf_counter() - t0

    def search(query, k=5):
        q = model.encode([query_prefix + query], normalize_embeddings=True,
                         convert_to_numpy=True, show_progress_bar=False).astype("float32")
        scores, ids = index.search(q, k)
        return [(int(i), float(s)) for i, s in zip(ids[0], scores[0]) if i != -1]

    stats = dict(model=EMBED_MODEL, vector_dim=vecs.shape[1], similarity="cosine (inner product of L2-normalised vectors)",
                 index="faiss.IndexFlatIP (exact search)", batch_size=batch_size, n_vectors=index.ntotal,
                 build_seconds=build_s, device=DEVICE)
    return search, stats

embedder.encode(["warm-up"] * 8, batch_size=8)   # first call pays CUDA / kernel start-up, keep it out of timings
print("dense search ready")

dense search ready


### A2 — Fixed-size chunking at 100, 400 and 500 tokens

The whole corpus is tokenized once with the bge tokenizer and cut into consecutive windows of *N* tokens with **no overlap**. Two details:
* Each chunk is a slice of the **original text**, located through the tokenizer's character offsets. Decoding the tokens back would lowercase everything, because the tokenizer is uncased.
* A window never ends in the middle of a word. If the *N*-th token is a `##` word-piece, the boundary moves back to the start of that word. Chunks therefore have **at most** *N* tokens; the actual lengths are reported below.

The chunker treats the corpus as one continuous text, so a chunk may span two short articles. That is what plain fixed-size chunking does.

In [33]:
def fixed_size_chunks(text, size, tok=emb_tok):
    enc = tok(text, add_special_tokens=False, return_offsets_mapping=True, verbose=False)
    pieces, offs = tok.convert_ids_to_tokens(enc["input_ids"]), enc["offset_mapping"]
    n, start, bounds = len(pieces), 0, []
    while start < n:
        end = min(start + size, n)
        while end < n and end > start + 1 and pieces[end].startswith("##"):
            end -= 1                                  # don't cut a word in half
        bounds.append((start, end)); start = end
    chunks = []
    for i, (s, e) in enumerate(bounds):
        c0, c1 = offs[s][0], (offs[e][0] if e < n else len(text))   # slice up to the next chunk: nothing is lost
        secs = sorted({bisect.bisect_right(section_starts, c0) - 1, bisect.bisect_right(section_starts, c1 - 1) - 1})
        ids = [sections[j]["id"].replace("Article ", "Art. ") for j in range(secs[0], secs[-1] + 1)]
        chunks.append(dict(id=i, text=text[c0:c1].strip(), start=c0, end=c1,
                           label=ids[0] if len(ids) == 1 else f"{ids[0]} … {ids[-1]}"))
    for c in chunks:
        c["n_tokens"] = len(tok(c["text"], add_special_tokens=False, verbose=False)["input_ids"])
    return chunks

def is_hit(chunk_text, span):
    return norm(span) in norm(chunk_text)

CHUNK_SIZES = [100, 400, 500]
chunk_sets, dense, dense_stats = {}, {}, {}
for size in CHUNK_SIZES:
    chunk_sets[size] = fixed_size_chunks(corpus, size)
    dense[size], dense_stats[size] = build_dense_search([c["text"] for c in chunk_sets[size]])
    lens = [c["n_tokens"] for c in chunk_sets[size]]
    assert max(lens) + 2 <= embedder.max_seq_length, "a chunk would be truncated by the embedder"
    over = sum(len(minilm_tok(c["text"], verbose=False)["input_ids"]) > 256 for c in chunk_sets[size])   # incl. [CLS]/[SEP]
    print(f"size {size:3d}: {len(lens):5d} chunks | tokens/chunk mean {np.mean(lens):.0f}, max {max(lens)} | "
          f"built in {dense_stats[size]['build_seconds']:.1f}s | would be truncated by MiniLM (256): {over} ({over/len(lens):.0%})")

size 100:  1537 chunks | tokens/chunk mean 100, max 100 | built in 11.4s | would be truncated by MiniLM (256): 0 (0%)
size 400:   384 chunks | tokens/chunk mean 400, max 400 | built in 11.5s | would be truncated by MiniLM (256): 384 (100%)
size 500:   308 chunks | tokens/chunk mean 498, max 500 | built in 11.9s | would be truncated by MiniLM (256): 307 (100%)


In [34]:
def hit_rank(chunk_ids, chunks, span):
    """1-based rank of the first retrieved chunk that contains the span, or None."""
    return next((r for r, cid in enumerate(chunk_ids, 1) if is_hit(chunks[cid]["text"], span)), None)

qwen_tok = AutoTokenizer.from_pretrained(json.loads((MODELS_DIR / "a1_model_info.json").read_text())["base_model"])
a2_rows, a2_detail = [], {}
for size in CHUNK_SIZES:
    chunks = chunk_sets[size]
    tops = [[cid for cid, _ in dense[size](q, 3)] for q in Q]
    ranks = [hit_rank(t, chunks, qq["span"]) for t, qq in zip(tops, queries)]
    a2_detail[size] = ranks
    reachable = sum(any(is_hit(c["text"], qq["span"]) for c in chunks) for qq in queries)
    a2_rows.append({
        "Chunk size (tokens)": size,
        "Total chunks": len(chunks),
        "Hit rate@3 (%)": 100 * sum(r is not None for r in ranks) / len(Q),
        "Tokens sent per query (top 3)": np.mean([sum(chunks[c]["n_tokens"] for c in t) for t in tops]),
        "…same, in Qwen3 (generator) tokens": np.mean([sum(len(qwen_tok(chunks[c]["text"])["input_ids"]) for c in t) for t in tops]),
        "Spans that fit whole in some chunk": f"{reachable}/10",
    })
a2_table = pd.DataFrame(a2_rows).set_index("Chunk size (tokens)")
display(a2_table.round(1))
display(pd.DataFrame({f"{s} tok": ["✓ rank " + str(r) if r else "✗" for r in a2_detail[s]] for s in CHUNK_SIZES},
                     index=[q["id"] for q in queries]).T)

,Total chunks,Hit rate@3 (%),Tokens sent per query (top 3),"…same, in Qwen3 (generator) tokens",Spans that fit whole in some chunk
Chunk size (tokens),,,,,
100,1537,80.0,299.7,307.8,9/10
400,384,90.0,1199.6,1229.5,9/10
500,308,80.0,1499.7,1540.3,10/10


,coi-011,coi-024,coi-058,coi-067,coi-070,coi-094,coi-129,coi-138,coi-221,coi-248
100 tok,✓ rank 1,✓ rank 1,✓ rank 1,✓ rank 1,✓ rank 1,✓ rank 1,✓ rank 1,✓ rank 1,✗,✗
400 tok,✓ rank 1,✓ rank 1,✓ rank 1,✓ rank 2,✓ rank 2,✓ rank 2,✓ rank 1,✓ rank 1,✓ rank 1,✗
500 tok,✓ rank 1,✓ rank 2,✓ rank 1,✗,✓ rank 2,✓ rank 1,✓ rank 1,✓ rank 1,✓ rank 2,✗


*"Spans that fit whole in some chunk"* is an upper bound on the hit rate. With no overlap, a span that falls across a chunk boundary can never be a hit, and small chunks have the most boundaries.

### A2.2 — Chosen chunk size

**We use 400-token chunks.**
* **Hit rate.** 400 has the best hit rate@3: 90%, against 80% for both 100 and 500.
  * 100-token chunks cut articles into fragments, and for coi-221 (the Tenth Schedule merger rule) the right fragment falls out of the top 3.
  * 500-token chunks pack several articles into one vector, which blurs the match, and coi-067 (Article 81) drops out of the top 3 entirely.
* **Tokens per query.** 400-token chunks send about 1,200 tokens per query, four times as many as 100-token chunks.
* **Generator context.** About 1,200 tokens is well within Qwen3-0.6B's 32k context window. It is longer than anything our Assignment 1 model saw in SFT (≤ 512 tokens), but 500-token chunks would add another 300 tokens for a *lower* hit rate.

In [35]:
check_quoted({s: a2_table.loc[s, "Hit rate@3 (%)"] for s in CHUNK_SIZES}, {100: 80, 400: 90, 500: 80}, "A2.2")
CHOSEN_CHUNK_SIZE = 400
chunks = chunk_sets[CHOSEN_CHUNK_SIZE]
dense_search = dense[CHOSEN_CHUNK_SIZE]
print(f"Parts B–D use {CHOSEN_CHUNK_SIZE}-token chunks ({len(chunks)} chunks)")

✓ the numbers quoted in A2.2 match this run
Parts B–D use 400-token chunks (384 chunks)


### A1.2 — Dense index report for the final chunk set

In [36]:
s = dense_stats[CHOSEN_CHUNK_SIZE]
a12 = pd.DataFrame({"Value": {
    "Embedding model": s["model"],
    "Max input length": f"{embedder.max_seq_length} tokens (bge tokenizer)",
    "Vector size": s["vector_dim"],
    "Similarity measure": s["similarity"],
    "Index": s["index"],
    "Batch size (embedding)": s["batch_size"],
    "Chunks indexed": f"{s['n_vectors']} × {CHOSEN_CHUNK_SIZE}-token chunks",
    "Index build time (embed + add)": f"{s['build_seconds']:.2f} s on {s['device']}",
}})
display(a12)

,Value
Embedding model,BAAI/bge-base-en-v1.5
Max input length,512 tokens (bge tokenizer)
Vector size,768
Similarity measure,cosine (inner product of L2-normalised vectors)
Index,faiss.IndexFlatIP (exact search)
Batch size (embedding),64
Chunks indexed,384 × 400-token chunks
Index build time (embed + add),11.47 s on cuda


## Part B — Dense vs keyword (BM25) vs hybrid (RRF)

All three methods use the same chunks and return the top 5.
* **Dense:** the Part A search.
* **BM25:** `rank_bm25.BM25Okapi` with default parameters (k1 = 1.5, b = 0.75). Tokens are lowercased alphanumeric runs, so article numbers such as `243zj` stay whole, and a short English stop-word list is removed.
* **Hybrid:** Reciprocal Rank Fusion of the **top 20** from each method: `RRF(d) = 1/(60 + rank_dense) + 1/(60 + rank_bm25)`. Ranks start at 1, and a chunk missing from one list gets nothing from that list. Ties are broken by dense rank.

**Latency** is the wall-clock time to return the top 5 for one query, including encoding the query for dense and hybrid, averaged over the 10 queries after one warm-up call.

In [37]:
STOPWORDS = set("""a an and are as at be by can do does for from has have how if in into is it its may of on or
shall should such that the their there these this those to under was were what when where which who whom
why will with would""".split())

def bm25_tokenize(t):
    return [w for w in re.findall(r"[a-z0-9]+", t.lower()) if w not in STOPWORDS]

bm25 = BM25Okapi([bm25_tokenize(c["text"]) for c in chunks])

def dense_ids(q, k):
    return [cid for cid, _ in dense_search(q, k)]

def bm25_ids(q, k):
    scores = bm25.get_scores(bm25_tokenize(q))
    return [int(i) for i in np.argsort(-scores, kind="stable")[:k]]

def rrf_fuse(lists, k_rrf=60):
    score, first_rank = collections.defaultdict(float), {}
    for li, ranked in enumerate(lists):
        for rank, cid in enumerate(ranked, start=1):
            score[cid] += 1.0 / (k_rrf + rank)
            first_rank.setdefault(cid, (li, rank))
    return sorted(score, key=lambda c: (-score[c], first_rank[c]))

def hybrid_ids(q, k, depth=20):
    return rrf_fuse([dense_ids(q, depth), bm25_ids(q, depth)])[:k]

METHODS = {"Dense": dense_ids, "BM25": bm25_ids, "Hybrid (RRF)": hybrid_ids}

def run_method(fn, k=5):
    fn(Q[0], k)                                      # warm-up, not timed
    results, times = [], []
    for q in Q:
        sync(); t0 = time.perf_counter()
        results.append(fn(q, k))
        sync(); times.append((time.perf_counter() - t0) * 1000)
    return results, float(np.mean(times))

retrieved, b_rows = {}, []
for name, fn in METHODS.items():
    retrieved[name], ms = run_method(fn)
    ranks = [hit_rank(r, chunks, qq["span"]) for r, qq in zip(retrieved[name], queries)]
    b_rows.append({"Method": name, "Average query latency (ms)": ms,
                   "Hit rate@1 (%)": 100 * sum(r == 1 for r in ranks) / len(Q),
                   "Hit rate@3 (%)": 100 * sum(r is not None and r <= 3 for r in ranks) / len(Q),
                   "Hit rate@5 (%)": 100 * sum(r is not None for r in ranks) / len(Q)})
b_table = pd.DataFrame(b_rows).set_index("Method")
display(b_table.round(2))
check_quoted({f"{m} {c}": b_table.loc[m, f"Hit rate@{c} (%)"] for m in METHODS for c in (1, 3)},
             {"Dense 1": 60, "Dense 3": 90, "BM25 1": 60, "BM25 3": 80, "Hybrid (RRF) 1": 70, "Hybrid (RRF) 3": 80}, "B4")

per_query = pd.DataFrame({name: [("rank " + str(r)) if (r := hit_rank(retrieved[name][i], chunks, qq["span"])) else "✗ (not in top 5)"
                                 for i, qq in enumerate(queries)] for name in METHODS},
                         index=[f"{qq['id']}: {qq['question'][:70]}" for qq in queries])
print("Rank of the first chunk containing the answer span:")
display(per_query)

,Average query latency (ms),Hit rate@1 (%),Hit rate@3 (%),Hit rate@5 (%)
Method,,,,
Dense,88.06,60.0,90.0,90.0
BM25,7.97,60.0,80.0,80.0
Hybrid (RRF),71.66,70.0,80.0,90.0


✓ the numbers quoted in B4 match this run
Rank of the first chunk containing the answer span:


,Dense,BM25,Hybrid (RRF)
coi-011: On which grounds does the Constitution forbid the State from discrimin,rank 1,rank 1,rank 1
coi-024: What does the Constitution say about employing children in factories?,rank 1,rank 1,rank 1
coi-058: Which court decides disputes about the election of the President or Vi,rank 1,rank 1,rank 1
coi-067: What is the maximum strength of the House of the People?,rank 2,✗ (not in top 5),rank 5
coi-070: What is the maximum permitted gap between two sessions of Parliament?,rank 2,rank 3,rank 1
coi-094: How is a judge of the Supreme Court removed?,rank 2,rank 1,rank 2
coi-129: What share of Panchayat seats is reserved for women?,rank 1,rank 2,rank 1
coi-138: What is the maximum size of the board of a co-operative society?,rank 1,rank 1,rank 1
coi-221: Is a merger of political parties treated as defection under the Tenth,rank 1,rank 1,rank 1
coi-248: A 27-year-old citizen wants to contest elections to both the Lok Sabha,✗ (not in top 5),✗ (not in top 5),✗ (not in top 5)


### B4 — Which method works best?

**Dense search works best for our queries on hit rate@3**: 90%, against 80% for BM25 and for hybrid. Hit rate@3 is what decides which chunks the generator sees, so we take dense into Part C. Hybrid has the best hit rate@1 (70%).

* **Example where dense wins: coi-067, "What is the *maximum strength* of the House of the People?"** The Constitution never uses the words "maximum" or "strength". It says the House "shall consist of not more than five hundred and thirty members…". BM25 can match only "house" and "people", which occur in hundreds of chunks, so the right chunk is not in its top 5 at all. Dense matches the meaning and ranks the chunk 2nd. In the fusion this chunk scores from the dense list only, so chunks that both lists rank moderately well push it down to 5th. That is why hybrid's hit rate@3 falls below dense here.
* **Where BM25 wins:** coi-094 (removing a Supreme Court judge). The exact words "Judge of the Supreme Court … removed" put Article 124 first for BM25. Dense ranks Article 217 first, which is about removing *High Court* judges "in the manner provided in … article 124".
* **Where all three fail:** coi-248. The query says "Lok Sabha / Rajya Sabha" but the corpus says "House of the People / Council of States". In addition, the answer span is split across two 400-token chunks, so no single chunk can be a hit.
* **Latency.** BM25 is more than an order of magnitude faster (under 1 ms, because it needs no neural encoding of the query). At this corpus size, though, every method answers in milliseconds, so speed does not decide the choice.

## Part C — Reranking with a cross-encoder

The best Part B method retrieves the **top 10** for each query. The cross-encoder **`cross-encoder/ms-marco-MiniLM-L-6-v2`** reads each (query, chunk) pair *together*, which is more accurate than comparing two independent embeddings. It rescores the 10 chunks and we keep the **top 3**. Its input limit is 512 tokens, the same as the embedder, so a 400-token chunk plus the query fits.

In [38]:
best = b_table.sort_values(["Hit rate@3 (%)", "Hit rate@1 (%)", "Hit rate@5 (%)"], ascending=False).index[0]
BEST_METHOD = best                                  # chosen from the Part B table
print("best method from Part B:", BEST_METHOD)

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2", max_length=512, device=DEVICE)
ce_tok = reranker.tokenizer

def rerank(q, cand_ids, keep=3):
    scores = reranker.predict([(q, chunks[c]["text"]) for c in cand_ids], batch_size=16, show_progress_bar=False)
    order = np.argsort(-np.asarray(scores), kind="stable")
    return [cand_ids[i] for i in order[:keep]], [float(scores[i]) for i in order[:keep]]

before10 = [METHODS[BEST_METHOD](q, 10) for q in Q]
rerank(Q[0], before10[0])                           # warm-up
after3, rr_ms = [], []
for q, cands in zip(Q, before10):
    sync(); t0 = time.perf_counter()
    ids, _ = rerank(q, cands)
    sync(); rr_ms.append((time.perf_counter() - t0) * 1000)
    after3.append(ids)

too_long = sum(len(ce_tok(q, chunks[c]["text"])["input_ids"]) > 512 for q, cands in zip(Q, before10) for c in cands)
print(f"(query, chunk) pairs longer than the cross-encoder's 512-token limit: {too_long} of {10 * len(Q)}")

best method from Part B: Dense


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

(query, chunk) pairs longer than the cross-encoder's 512-token limit: 0 of 100


In [39]:
def show_ids(ids, span):
    return " | ".join(f"#{c} {chunks[c]['label']}{' ✓' if is_hit(chunks[c]['text'], span) else ''}" for c in ids)

compare = pd.DataFrame([{
    "query": f"{qq['id']}: {qq['question'][:60]}",
    f"top 3 before ({BEST_METHOD})": show_ids(b[:3], qq["span"]),
    "top 3 after reranking": show_ids(a, qq["span"]),
    "first result changed": "yes" if b[0] != a[0] else "no",
} for qq, b, a in zip(queries, before10, after3)])
print("✓ = chunk contains the answer span;  #id = chunk id;  label = article(s) the chunk comes from")
display(compare)

hit1_before = 100 * sum(is_hit(chunks[b[0]]["text"], qq["span"]) for b, qq in zip(before10, queries)) / len(Q)
hit1_after = 100 * sum(is_hit(chunks[a[0]]["text"], qq["span"]) for a, qq in zip(after3, queries)) / len(Q)
hit3_before = 100 * sum(any(is_hit(chunks[c]["text"], qq["span"]) for c in b[:3]) for b, qq in zip(before10, queries)) / len(Q)
hit3_after = 100 * sum(any(is_hit(chunks[c]["text"], qq["span"]) for c in a) for a, qq in zip(after3, queries)) / len(Q)
rank_change = 100 * sum(b[0] != a[0] for b, a in zip(before10, after3)) / len(Q)
assert abs(hit1_before - b_table.loc[BEST_METHOD, "Hit rate@1 (%)"]) < 1e-9

c_table = pd.DataFrame({"Value": {
    "Reranking time per query (ms)": f"{np.mean(rr_ms):.1f}  (10 candidates, {DEVICE})",
    "Rank change rate (%)": f"{rank_change:.0f}",
    f"Hit rate@1 before reranking (from Part B, {BEST_METHOD})": f"{hit1_before:.0f}%",
    "Hit rate@1 after reranking": f"{hit1_after:.0f}%",
    "(extra) Hit rate@3 before → after": f"{hit3_before:.0f}% → {hit3_after:.0f}%",
}})
display(c_table)
check_quoted({"hit1_before": hit1_before, "hit1_after": hit1_after, "hit3_before": hit3_before,
              "hit3_after": hit3_after, "rank_change": rank_change},
             {"hit1_before": 60, "hit1_after": 70, "hit3_before": 90, "hit3_after": 80, "rank_change": 30}, "C4")

✓ = chunk contains the answer span;  #id = chunk id;  label = article(s) the chunk comes from


,query,top 3 before (Dense),top 3 after reranking,first result changed
0,coi-011: On which grounds does the Constitution forbid the State from,#5 Art. 13 … Art. 15 ✓ | #14 Art. 23 … Art. 26 | #7 Art. 16,#5 Art. 13 … Art. 15 ✓ | #7 Art. 16 | #14 Art. 23 … Art. 26,no
1,coi-024: What does the Constitution say about employing children in f,#14 Art. 23 … Art. 26 ✓ | #24 Art. 43 … Art. 47 | #11 Art. 21A … Art. 22,#14 Art. 23 … Art. 26 ✓ | #24 Art. 43 … Art. 47 | #22 Art. 37 … Art. 39A,no
2,coi-058: Which court decides disputes about the election of the Presi,#35 Art. 69 … Art. 71 ✓ | #34 Art. 66 … Art. 69 | #36 Art. 71 … Art. 73,#35 Art. 69 … Art. 71 ✓ | #33 Art. 65 … Art. 66 | #69 Art. 128 … Art. 131,no
3,coi-067: What is the maximum strength of the House of the People?,#238 Art. 352 | #40 Art. 79 … Art. 81 ✓ | #46 Art. 92 … Art. 95,#41 Art. 81 … Art. 82 | #216 Art. 330 … Art. 330A | #221 Art. 334A … Art. 336,yes
4,coi-070: What is the maximum permitted gap between two sessions of Pa,#50 Art. 101 … Art. 102 | #43 Art. 84 … Art. 87 ✓ | #42 Art. 82 … Art. 84,#43 Art. 84 … Art. 87 ✓ | #91 Art. 173 … Art. 176 | #50 Art. 101 … Art. 102,yes
5,coi-094: How is a judge of the Supreme Court removed?,#115 Art. 217 | #65 Art. 124 ✓ | #199 Art. 310 … Art. 311,#115 Art. 217 | #65 Art. 124 ✓ | #66 Art. 124 … Art. 124B,no
6,coi-129: What share of Panchayat seats is reserved for women?,#136 Art. 243D … Art. 243E ✓ | #135 Art. 243C … Art. 243D | #145 Art. 243S … Art. 243T,#136 Art. 243D … Art. 243E ✓ | #135 Art. 243C … Art. 243D | #217 Art. 330A … Art. 332,no
7,coi-138: What is the maximum size of the board of a co-operative soci,#154 Art. 243ZJ ✓ | #153 Art. 243ZH … Art. 243ZJ | #157 Art. 243ZM … Art. 243ZP,#154 Art. 243ZJ ✓ | #153 Art. 243ZH … Art. 243ZJ | #156 Art. 243ZL … Art. 243ZM,no
8,coi-221: Is a merger of political parties treated as defection under,#369 Tenth Schedule ✓ | #367 Ninth Schedule … Tenth Schedule | #249 Art. 361A … Art. 361B,#369 Tenth Schedule ✓ | #367 Ninth Schedule … Tenth Schedule | #85 Art. 164 … Art. 165,no
9,coi-248: A 27-year-old citizen wants to contest elections to both the,#90 Art. 171 … Art. 173 | #50 Art. 101 … Art. 102 | #214 Art. 325 … Art. 328,#33 Art. 65 … Art. 66 | #43 Art. 84 … Art. 87 | #214 Art. 325 … Art. 328,yes


,Value
Reranking time per query (ms),"87.5 (10 candidates, cuda)"
Rank change rate (%),30
"Hit rate@1 before reranking (from Part B, Dense)",60%
Hit rate@1 after reranking,70%
(extra) Hit rate@3 before → after,90% → 80%


✓ the numbers quoted in C4 match this run


### C4 — Did reranking help?

**Reranking gave a mixed result.** Hit rate@1 rose from **60% to 70%**, but hit rate@3, which is what the generator actually receives, fell from **90% to 80%**. The cross-encoder changed the first result for 30% of the queries.
* **Gain: coi-070.** It moved the Article 85 chunk, which holds the six-month rule, from rank 2 to rank 1, above an Article 101 chunk.
* **Loss: coi-067.** It promoted the second half of Article 81 (#41, about how seats are allotted, which repeats "House of the People") over #40, the chunk that actually states the 530 + 20 member cap. #40 dropped out of the top 3.
* **Cost.** The time it adds (see the table) is small next to answer generation, which takes seconds per query.

Overall, reranking helps top-1 precision a little in our domain but does not improve the context sent to the generator. The retrieval errors that remain come from **chunking** (coi-248's answer is cut across two chunks) and from **wording mismatch**, which re-ordering the same candidates cannot fix. We keep the reranked top 3 for Part D, as the brief asks.

## Part D1 — Grounded answer generation

**Generator.** We first try the **Assignment 1 model**: `cpt-merged` (Qwen3-0.6B-Base after continual pre-training), loaded in 4-bit, with the SFT adapter on top. Its RAG prompt keeps the `### System / ### Question / ### Answer` layout the model was fine-tuned on, and adds a `### Context:` block holding the **top 3 chunks after reranking**, numbered [1]–[3]. If the model does not follow the instructions, the brief allows switching to an instruct model; the check further down makes that decision. Decoding is greedy (`do_sample=False`, the same as temperature 0).

**The prompt** asks for one to three full sentences, each ending with the number of the chunk it came from. It gives a worked example about an article that is not among our queries, and says never to reply with chunk numbers alone. We added that rule after our first run: with the shorter instruction *"Cite the number of every chunk you used, for example [2]"*, Qwen2.5-7B-Instruct replied to 6 of the 10 questions with nothing but a chunk number such as `[2]`.

**Not-in-corpus queries.** The main one is *"Which party won the most seats in the 2024 Lok Sabha election?"* Election results are not in the Constitution, but a model may "know" the answer from pre-training, so this checks whether it stays grounded. We add a second, harder one: *"What is the GST rate charged on restaurant food?"* GST appears in the Constitution, so retrieval returns GST chunks that look relevant, but no rates are given anywhere.

In [40]:
RAG_RULES = ("Answer the question using ONLY the information in the numbered chunks given in the context. "
             "Write the answer as one to three complete sentences in your own words. "
             "End each sentence with the number of the chunk it comes from in square brackets, for example: "
             "\"The President is elected by an electoral college [1].\" "
             "Never reply with only chunk numbers. "
             "If none of the chunks contains the answer, reply with exactly: Not found in the documents")
A1_INFO = json.loads((MODELS_DIR / "a1_model_info.json").read_text())

def a1_prompt(question, chunk_texts):
    context = "\n\n".join(f"[{i}] {t}" for i, t in enumerate(chunk_texts, 1))
    return (f"### System:\nYou are an expert on the Constitution of India. {RAG_RULES}\n\n"
            f"### Context:\n{context}\n\n### Question:\n{question}\n\n### Answer:\n")

OOD_QUERIES = ["Which party won the most seats in the 2024 Lok Sabha election?",
               "What is the GST rate charged on restaurant food?"]
all_questions = Q + OOD_QUERIES
ctx_ids = after3 + [rerank(q, METHODS[BEST_METHOD](q, 10))[0] for q in OOD_QUERIES]
ctx_texts = [[chunks[c]["text"] for c in ids] for ids in ctx_ids]
print("The prompt (shown for query 1, chunk text shortened here only):\n")
print(a1_prompt(Q[0], [t[:150] + " …" for t in ctx_texts[0]]))

The prompt (shown for query 1, chunk text shortened here only):

### System:
You are an expert on the Constitution of India. Answer the question using ONLY the information in the numbered chunks given in the context. Write the answer as one to three complete sentences in your own words. End each sentence with the number of the chunk it comes from in square brackets, for example: "The President is elected by an electoral college [1]." Never reply with only chunk numbers. If none of the chunks contains the answer, reply with exactly: Not found in the documents

### Context:
[1] or abridges the rights conferred by this Part and any law made in contravention of this clause shall, to the extent of the contravention, be void.
(3) …

[2] — Fundamental Rights
Article 16: Equality of opportunity in matters of public employment

16. Equality of opportunity in matters of public employment. …

[3] of forced labour are prohibited and any contravention of this provision shall be an offence punishabl

In [41]:
MAX_NEW = 48 if SMOKE_TEST else 220

def load_causal_lm(path, quantize=HAS_GPU):
    kw = {DTYPE_KW: COMPUTE_DTYPE}
    if quantize:
        kw["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                                       bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=COMPUTE_DTYPE)
        kw["device_map"] = {"": 0}
    return AutoModelForCausalLM.from_pretrained(path, token=HF_TOKEN, **kw)

@torch.no_grad()
def generate(model, tok, prompts, max_new_tokens=MAX_NEW, batch_size=4, **gen_kw):
    tok.padding_side = "left"
    outs = []
    for i in range(0, len(prompts), batch_size):
        enc = tok(prompts[i:i + batch_size], return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)
        gen = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False, temperature=None, top_p=None,
                             top_k=None, pad_token_id=tok.pad_token_id, **gen_kw)
        outs += [tok.decode(row, skip_special_tokens=True).strip() for row in gen[:, enc["input_ids"].shape[1]:]]
    return outs

a1_tok = AutoTokenizer.from_pretrained(A1_INFO["cpt_merged_dir"])
a1_tok.pad_token = a1_tok.pad_token or a1_tok.eos_token
a1_model = PeftModel.from_pretrained(load_causal_lm(A1_INFO["cpt_merged_dir"]), A1_INFO["sft_adapter_dir"]).eval()
t0 = time.time()
a1_raw = generate(a1_model, a1_tok, [a1_prompt(q, c) for q, c in zip(all_questions, ctx_texts)],
                  repetition_penalty=1.1, eos_token_id=a1_tok.eos_token_id)
a1_answers = [re.split(r"\n\s*###", a)[0].strip() for a in a1_raw]     # stop if it starts a new "### …" block
print(f"generated {len(a1_answers)} answers in {time.time() - t0:.0f}s")
del a1_model; free_memory()

def says_not_found(a):
    return "not found in the documents" in a.lower()

def cites(a):
    return bool(re.search(r"\[[123]\]", a))

def is_bare(a):
    """True if the answer is only chunk numbers / punctuation, i.e. it makes no statement at all."""
    return not says_not_found(a) and len(re.findall(r"[A-Za-z]{2,}", re.sub(r"\[\d+\]", "", a))) < 3

for i, (q, a) in enumerate(zip(all_questions, a1_answers), 1):
    tag = "OUT-OF-CORPUS " if i > len(Q) else ""
    print(f"── {tag}Q{i}: {q}\n{a}\n")

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

generated 12 answers in 33s
── Q1: On which grounds does the Constitution forbid the State from discriminating against citizens under Article 15?
Article 26 forbids the State from discriminating against citizens on grounds only of religion, race, caste, sex, place of birth or any of them. It also prohibits the State from treating any class or classes of people differently because of their religion, race, caste, sex, descent, place of birth or residence.

── Q2: What does the Constitution say about employing children in factories?
Article 24 prohibits the employment of children under sixteen years old in factories, mines or any other hazardous work. It also forbids discriminating practices based on religion, race, caste or sex, and requires the State to ensure that no child is left behind in primary education.

── Q3: Which court decides disputes about the election of the President or Vice-President?
Article 66 requires that no doubt or dispute arising out of or in connection with the e

### Does the Assignment 1 model follow the RAG instructions?

The brief lets us switch to an instruct model if the Assignment 1 model ignores the chunks or never says "Not found". The cell below checks this automatically:
* Does it **cite** chunk numbers in at least half of the 10 answers?
* Does it reply **"Not found in the documents"** to the main not-in-corpus query?
* Does it **not** say "Not found" for in-corpus questions whose answer span was retrieved?

If any check fails, we show one example of the problem and generate the final answers with **Qwen2.5-7B-Instruct** (4-bit) instead.

In [42]:
cite_rate = sum(cites(a) for a in a1_answers[:len(Q)]) / len(Q)
ood_ok = says_not_found(a1_answers[len(Q)])
retrieved_ok = [any(is_hit(t, qq["span"]) for t in ctx) for qq, ctx in zip(queries, ctx_texts)]
false_nf = sum(says_not_found(a) and ok for a, ok in zip(a1_answers[:len(Q)], retrieved_ok))
bare = sum(is_bare(a) for a in a1_answers[:len(Q)])
print(f"answers with a [n] citation: {cite_rate:.0%} | 'Not found' on the out-of-corpus query: {ood_ok} | "
      f"'Not found' although the span was retrieved: {false_nf}/10 | only chunk numbers: {bare}/10")

FORCE_GENERATOR = None          # None = decide automatically; or "a1" / "instruct"
SWITCH = (cite_rate < 0.5 or not ood_ok or false_nf > 2 or bare > 2) if FORCE_GENERATOR is None else FORCE_GENERATOR == "instruct"

if SWITCH:
    if not ood_ok:
        i, why = len(Q), "it did not reply 'Not found in the documents' to a question the corpus cannot answer"
    elif cite_rate < 0.5:
        i, why = next(j for j, a in enumerate(a1_answers[:len(Q)]) if not cites(a)), "it ignored the instruction to cite chunks"
    else:
        i, why = next(j for j, (a, ok) in enumerate(zip(a1_answers, retrieved_ok)) if says_not_found(a) and ok), \
                 "it said 'Not found' although the answer was in the chunks"
    display(Markdown(f"**Problem seen with the Assignment 1 model:** {why}.\n\n"
                     f"> **Q:** {all_questions[i]}\n>\n> **A1 model:** {a1_answers[i]}"))
else:
    print("The Assignment 1 model follows the instructions, so we keep it as the generator.")

answers with a [n] citation: 0% | 'Not found' on the out-of-corpus query: False | 'Not found' although the span was retrieved: 0/10 | only chunk numbers: 0/10


**Problem seen with the Assignment 1 model:** it did not reply 'Not found in the documents' to a question the corpus cannot answer.

> **Q:** Which party won the most seats in the 2024 Lok Sabha election?
>
> **A1 model:** Under Article 56, the winning party was the People's Democratic Party of India (PDP-I), led by Narendra Modi, who secured forty-one seats out of fifty-nine contested ones.

In [43]:
GENERATOR_INSTRUCT = "Qwen/Qwen2.5-0.5B-Instruct" if SMOKE_TEST else "Qwen/Qwen2.5-7B-Instruct"

def chat_prompt(tok, system, user):
    msgs = [{"role": "system", "content": system}, {"role": "user", "content": user}]
    try:
        return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    except Exception:          # some chat templates have no system role: put the instructions at the top of the user turn
        return tok.apply_chat_template([{"role": "user", "content": f"{system}\n\n{user}"}],
                                       tokenize=False, add_generation_prompt=True)

INSTRUCT_SYSTEM = f"You are an assistant that answers questions about the Constitution of India. {RAG_RULES}"
def instruct_user(question, chunk_texts):
    context = "\n\n".join(f"[{i}] {t}" for i, t in enumerate(chunk_texts, 1))
    return f"Context:\n{context}\n\nQuestion: {question}"

if SWITCH:
    GENERATOR = GENERATOR_INSTRUCT
    gen_tok = AutoTokenizer.from_pretrained(GENERATOR, token=HF_TOKEN)
    gen_tok.pad_token = gen_tok.pad_token or gen_tok.eos_token
    gen_model = load_causal_lm(GENERATOR).eval()
    t0 = time.time()
    answers = generate(gen_model, gen_tok, [chat_prompt(gen_tok, INSTRUCT_SYSTEM, instruct_user(q, c))
                                            for q, c in zip(all_questions, ctx_texts)])
    print(f"{GENERATOR}: {len(answers)} answers in {time.time() - t0:.0f}s")
    del gen_model; free_memory()
    print("\nThe prompt (system + user message; chunk text shortened here only):\n")
    print(chat_prompt(gen_tok, INSTRUCT_SYSTEM, instruct_user(Q[0], [t[:150] + " …" for t in ctx_texts[0]])))
else:
    GENERATOR, answers = "Assignment 1 model (Qwen3-0.6B-Base + CPT + SFT)", a1_answers
print("\nGENERATOR USED FOR THE FINAL ANSWERS:", GENERATOR)

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Qwen/Qwen2.5-7B-Instruct: 12 answers in 48s

The prompt (system + user message; chunk text shortened here only):

<|im_start|>system
You are an assistant that answers questions about the Constitution of India. Answer the question using ONLY the information in the numbered chunks given in the context. Write the answer as one to three complete sentences in your own words. End each sentence with the number of the chunk it comes from in square brackets, for example: "The President is elected by an electoral college [1]." Never reply with only chunk numbers. If none of the chunks contains the answer, reply with exactly: Not found in the documents<|im_end|>
<|im_start|>user
Context:
[1] or abridges the rights conferred by this Part and any law made in contravention of this clause shall, to the extent of the contravention, be void.
(3) …

[2] — Fundamental Rights
Article 16: Equality of opportunity in matters of public employment

16. Equality of opportunity in matters of public employment. …

In [44]:
d1 = pd.DataFrame({"query": [f"{qq['id']}: {qq['question']}" for qq in queries] + [f"OUT-OF-CORPUS: {q}" for q in OOD_QUERIES],
                   "chunks given": [show_ids(ids, qq["span"]) for ids, qq in zip(ctx_ids, queries)] +
                                   [" | ".join(f"#{c} {chunks[c]['label']}" for c in ids) for ids in ctx_ids[len(Q):]],
                   "answer": answers})
display(d1)
print("All answers in full:\n")
for k, (q, a) in enumerate(zip(all_questions, answers), 1):
    print(f"── {'OUT-OF-CORPUS ' if k > len(Q) else ''}Q{k}: {q}\n{a}\n")
for q, a in zip(OOD_QUERIES, answers[len(Q):]):
    print(f"Out-of-corpus: {q}\n  → {a}\n  → replied 'Not found': {says_not_found(a)}")

bare_ids = [qq["id"] for qq, a in zip(queries, answers) if is_bare(a)]
wrong_nf = [qq["id"] for qq, a, ok in zip(queries, answers, retrieved_ok) if says_not_found(a) and ok]
print(f"\nQuality check: answers that are only chunk numbers: {bare_ids or 'none'} | "
      f"'Not found' although the answer span was in the chunks: {wrong_nf or 'none'}")

,query,chunks given,answer
0,coi-011: On which grounds does the Constitution forbid the State from discriminating against citizens under Article 15?,#5 Art. 13 … Art. 15 ✓ | #7 Art. 16 | #14 Art. 23 … Art. 26,"The Constitution forbids the State from discriminating against citizens on grounds only of religion, race, caste, sex, place of birth, or any of them under Article 15 [2]."
1,coi-024: What does the Constitution say about employing children in factories?,#14 Art. 23 … Art. 26 ✓ | #24 Art. 43 … Art. 47 | #22 Art. 37 … Art. 39A,The Constitution of India prohibits the employment of children below the age of fourteen years in any factory or mine or in any other hazardous employment [1].
2,coi-058: Which court decides disputes about the election of the President or Vice-President?,#35 Art. 69 … Art. 71 ✓ | #33 Art. 65 … Art. 66 | #69 Art. 128 … Art. 131,All doubts and disputes arising out of or in connection with the election of a President or Vice-President shall be inquired into and decided by the Supreme Court whose decision shall be final [1].
3,coi-067: What is the maximum strength of the House of the People?,#41 Art. 81 … Art. 82 | #216 Art. 330 … Art. 330A | #221 Art. 334A … Art. 336,Not found in the documents
4,coi-070: What is the maximum permitted gap between two sessions of Parliament?,#43 Art. 84 … Art. 87 ✓ | #91 Art. 173 … Art. 176 | #50 Art. 101 … Art. 102,Six months shall not intervene between its last sitting in one session and the date appointed for its first sitting in the next session [1].
5,coi-094: How is a judge of the Supreme Court removed?,#115 Art. 217 | #65 Art. 124 ✓ | #66 Art. 124 … Art. 124B,A judge of the Supreme Court can be removed from office except by an order of the President passed after an address by each House of Parliament supported by a majority of the total membership of that House and by a majority of not less than two-thirds of the members of that House present and voting has been presented to the President in the same session for such removal on the ground of proved misbehaviour or incapacity [4].
6,coi-129: What share of Panchayat seats is reserved for women?,#136 Art. 243D … Art. 243E ✓ | #135 Art. 243C … Art. 243D | #217 Art. 330A … Art. 332,Not found in the documents
7,coi-138: What is the maximum size of the board of a co-operative society?,#154 Art. 243ZJ ✓ | #153 Art. 243ZH … Art. 243ZJ | #156 Art. 243ZL … Art. 243ZM,The maximum size of the board of a co-operative society is twenty-one members [1].
8,coi-221: Is a merger of political parties treated as defection under the Tenth Schedule?,#369 Tenth Schedule ✓ | #367 Ninth Schedule … Tenth Schedule | #85 Art. 164 … Art. 165,"No, a merger of political parties is not treated as defection under the Tenth Schedule. Under Article 102(2)(1)(a) and (b), a member is disqualified for defection if they voluntarily give up their membership or vote against the direction of their political party. However, if a member's original political party merges with another party, they are not disqualified under these provisions [1]."
9,coi-248: A 27-year-old citizen wants to contest elections to both the Lok Sabha and the Rajya Sabha. Are they eligible for both?,#33 Art. 65 … Art. 66 | #43 Art. 84 … Art. 87 | #214 Art. 325 … Art. 328,"Yes, a 27-year-old citizen is eligible to contest elections to both the Lok Sabha and the Rajya Sabha as there is no minimum age specified for the Lok Sabha that exceeds the eligibility for the Rajya Sabha. [2]"


All answers in full:

── Q1: On which grounds does the Constitution forbid the State from discriminating against citizens under Article 15?
The Constitution forbids the State from discriminating against citizens on grounds only of religion, race, caste, sex, place of birth, or any of them under Article 15 [2].

── Q2: What does the Constitution say about employing children in factories?
The Constitution of India prohibits the employment of children below the age of fourteen years in any factory or mine or in any other hazardous employment [1].

── Q3: Which court decides disputes about the election of the President or Vice-President?
All doubts and disputes arising out of or in connection with the election of a President or Vice-President shall be inquired into and decided by the Supreme Court whose decision shall be final [1].

── Q4: What is the maximum strength of the House of the People?
Not found in the documents

── Q5: What is the maximum permitted gap between two sessions of Pa

## Part D2 — Checking the answers with an LLM judge

The judge must be a **different model from the generator**, preferably from another model family so that it does not share the generator's blind spots. We use the first model in this list that the account can download:
1. `meta-llama/Llama-3.1-8B-Instruct`. Gated: it needs approved access on Hugging Face.
2. `microsoft/phi-4` (14B, MIT licence, not gated). This is the strongest open option that still fits a T4 in 4-bit (about 9 GB of GPU memory). The download is about 29 GB, so the first load takes several minutes.
3. `mistralai/Mistral-7B-Instruct-v0.3` (Apache 2.0, not gated). Smaller and faster than phi-4 (about 14 GB download).
4. `ibm-granite/granite-3.3-8b-instruct` (open). In our first run it sometimes graded the *expected* answer instead of the model's answer, so it comes last.

`Qwen/Qwen2.5-7B-Instruct` can be the judge only when the generator is the Assignment 1 model. Set `JUDGE_OVERRIDE` to force a specific model. The judge runs in 4-bit with greedy decoding (temperature 0), so the same input always gets the same verdict.

In [45]:
from huggingface_hub import hf_hub_download

def accessible(repo):
    try:
        hf_hub_download(repo, "config.json", token=HF_TOKEN)
        return True
    except Exception as e:
        print(f"  {repo}: not available ({type(e).__name__})")
        return False

if SMOKE_TEST:
    JUDGE_CANDIDATES = ["meta-llama/Llama-3.2-1B-Instruct", "HuggingFaceTB/SmolLM2-360M-Instruct"]
else:
    JUDGE_CANDIDATES = ["meta-llama/Llama-3.1-8B-Instruct", "microsoft/phi-4", "mistralai/Mistral-7B-Instruct-v0.3",
                        "Qwen/Qwen2.5-7B-Instruct", "ibm-granite/granite-3.3-8b-instruct"]
JUDGE_OVERRIDE = None        # e.g. "mistralai/Mistral-7B-Instruct-v0.3" to force a model
if JUDGE_OVERRIDE:
    assert JUDGE_OVERRIDE != GENERATOR, "the judge must differ from the generator"
    JUDGE = JUDGE_OVERRIDE
else:
    JUDGE = next(m for m in JUDGE_CANDIDATES if m != GENERATOR and accessible(m))
print("JUDGE MODEL:", JUDGE, "| generator:", GENERATOR)

  meta-llama/Llama-3.1-8B-Instruct: not available (GatedRepoError)
JUDGE MODEL: microsoft/phi-4 | generator: Qwen/Qwen2.5-7B-Instruct


In [46]:
JUDGE_SYSTEM = ("You are a strict examiner grading answers about the Constitution of India. "
                "Follow the marking scheme exactly and reply only in the required format.")

JUDGE_TEMPLATE = """You will grade one MODEL ANSWER to a question about the Constitution of India.

<question>
{question}
</question>

<expected_answer>
{expected}
</expected_answer>

<retrieved_chunks>
{chunks}
</retrieved_chunks>

<model_answer>
{answer}
</model_answer>

Grade ONLY the text inside <model_answer>. The expected answer and the chunks are reference material: never give the model answer credit for a fact that appears only in them.

Check 1 - CORRECT. Compare the facts stated in <model_answer> with <expected_answer>. Ignore wording, length and chunk numbers such as [2].
- correct: states the key facts of the expected answer and nothing that contradicts it.
- partly_correct: states at least one key fact of the expected answer but leaves out another key fact, or adds a wrong detail next to correct key facts.
- wrong: states none of the key facts, or contradicts the main point of the expected answer.
- If <model_answer> is "Not found in the documents", or makes no factual statement at all (for example it is only a chunk number such as [2]), it is wrong.

Check 2 - SUPPORTED. Ignore <expected_answer>. Check every factual statement in <model_answer> against <retrieved_chunks>.
- yes: every statement appears in, or follows directly from, the retrieved chunks.
- no: at least one statement is not in the retrieved chunks, even if it happens to be true.
- If <model_answer> makes no factual statement ("Not found in the documents", or only chunk numbers), it is supported: yes.

Reply with exactly these five lines and nothing else:
ANSWER_SAYS: <one short sentence restating what the model answer claims, or "no claim">
CORRECT: <correct | partly_correct | wrong>
SUPPORTED: <yes | no>
REASON: <one sentence explaining both verdicts>
UNSUPPORTED: <exact quote of the first unsupported statement, or none>"""

def judge_prompt(qq, chunk_texts, answer):
    return JUDGE_TEMPLATE.format(question=qq["question"], expected=qq["expected"], answer=answer,
                                 chunks="\n\n".join(f"[{i}] {t}" for i, t in enumerate(chunk_texts, 1)))

def parse_verdict(text):
    def field(key):
        m = re.search(rf"^\W*{key}\W*:\s*(.+)$", text, flags=re.I | re.M)
        return m.group(1).strip() if m else ""
    c, s = field("CORRECT").strip("*<> .").lower(), field("SUPPORTED").strip("*<> .").lower()
    coerced = c.startswith(("yes", "no"))          # judge answered yes/no instead of a label
    correct = ("partly_correct" if c.startswith(("partly", "partial")) else "correct" if c.startswith(("correct", "yes"))
               else "wrong" if c.startswith(("wrong", "incorrect", "no")) else "unparsed")
    supported = "yes" if s.startswith("yes") else "no" if s.startswith("no") else "unparsed"
    return dict(answer_says=field("ANSWER_SAYS"), correct=correct, supported=supported,
                reason=field("REASON"), unsupported=field("UNSUPPORTED"), coerced=coerced)

print("JUDGE PROMPT (system):\n" + JUDGE_SYSTEM + "\n\nJUDGE PROMPT (user):\n" + JUDGE_TEMPLATE)

JUDGE PROMPT (system):
You are a strict examiner grading answers about the Constitution of India. Follow the marking scheme exactly and reply only in the required format.

JUDGE PROMPT (user):
You will grade one MODEL ANSWER to a question about the Constitution of India.

<question>
{question}
</question>

<expected_answer>
{expected}
</expected_answer>

<retrieved_chunks>
{chunks}
</retrieved_chunks>

<model_answer>
{answer}
</model_answer>

Grade ONLY the text inside <model_answer>. The expected answer and the chunks are reference material: never give the model answer credit for a fact that appears only in them.

Check 1 - CORRECT. Compare the facts stated in <model_answer> with <expected_answer>. Ignore wording, length and chunk numbers such as [2].
- correct: states the key facts of the expected answer and nothing that contradicts it.
- partly_correct: states at least one key fact of the expected answer but leaves out another key fact, or adds a wrong detail next to correct key fac

In [47]:
judge_tok = AutoTokenizer.from_pretrained(JUDGE, token=HF_TOKEN)
judge_tok.pad_token = judge_tok.pad_token or judge_tok.eos_token
judge_model = load_causal_lm(JUDGE).eval()
t0 = time.time()
judge_raw = generate(judge_model, judge_tok,
                     [chat_prompt(judge_tok, JUDGE_SYSTEM, judge_prompt(qq, ctx, a))
                      for qq, ctx, a in zip(queries, ctx_texts, answers[:len(Q)])],
                     max_new_tokens=48 if SMOKE_TEST else 220, batch_size=2)
print(f"judged {len(judge_raw)} answers in {time.time() - t0:.0f}s")
del judge_model; free_memory()

verdicts = [parse_verdict(t) for t in judge_raw]
judge_table = pd.DataFrame([{"query": f"{qq['id']}: {qq['question']}", "judge read the answer as": v["answer_says"],
                             "correct": v["correct"], "supported": v["supported"],
                             "reason": v["reason"], "unsupported quote": v["unsupported"]}
                            for qq, v in zip(queries, verdicts)])
display(judge_table)
summary = pd.DataFrame({"Count out of 10": {
    "Correct": sum(v["correct"] == "correct" for v in verdicts),
    "Partly correct": sum(v["correct"] == "partly_correct" for v in verdicts),
    "Wrong": sum(v["correct"] == "wrong" for v in verdicts),
    "Supported by the chunks": sum(v["supported"] == "yes" for v in verdicts),
}})
display(summary)
if any(v["coerced"] for v in verdicts):
    print("⚠ the judge wrote yes/no instead of a CORRECT label for:", [qq["id"] for qq, v in zip(queries, verdicts) if v["coerced"]],
          "(read as yes → correct, no → wrong)")
if any("unparsed" in (v["correct"], v["supported"]) for v in verdicts):
    print("⚠ some verdicts could not be parsed; raw judge output:")
    for t in judge_raw:
        print("---\n" + t)

Loading weights:   0%|          | 0/243 [00:00<?, ?it/s]

judged 10 answers in 116s


,query,judge read the answer as,correct,supported,reason,unsupported quote
0,coi-011: On which grounds does the Constitution forbid the State from discriminating against citizens under Article 15?,"The Constitution forbids the State from discriminating against citizens on grounds only of religion, race, caste, sex, place of birth, or any of them under Article 15.",correct,yes,The model answer accurately states the grounds of discrimination prohibited by Article 15 as per the retrieved chunks.,none
1,coi-024: What does the Constitution say about employing children in factories?,The Constitution of India prohibits the employment of children below the age of fourteen years in any factory or mine or in any other hazardous employment.,correct,yes,The model answer accurately states the prohibition as outlined in Article 24 of the Constitution.,none
2,coi-058: Which court decides disputes about the election of the President or Vice-President?,The Supreme Court decides disputes about the election of the President or Vice-President.,correct,yes,"The model answer accurately states that the Supreme Court has the exclusive authority to decide disputes related to the election of the President or Vice-President, as per Article 71.",none
3,coi-067: What is the maximum strength of the House of the People?,Not found in the documents,wrong,yes,"The model answer makes no factual statement, which is incorrect as the information is available in the documents.",none
4,coi-070: What is the maximum permitted gap between two sessions of Parliament?,Six months shall not intervene between the last sitting of one session and the first sitting of the next session of Parliament.,correct,yes,"The model answer accurately states the requirement from Article 85 of the Constitution of India, and this information is supported by the retrieved chunks.",none
5,coi-094: How is a judge of the Supreme Court removed?,A judge of the Supreme Court can be removed by a presidential order after both Houses of Parliament pass an address with the required majority on the grounds of proved misbehaviour or incapacity.,correct,yes,The model answer accurately states the process and grounds for removal as outlined in Article 124(4) of the Constitution.,none
6,coi-129: What share of Panchayat seats is reserved for women?,Not found in the documents,wrong,yes,"The model answer makes no factual statement, which contradicts the expected answer that provides specific details about the reservation of Panchayat seats for women.",none
7,coi-138: What is the maximum size of the board of a co-operative society?,The maximum size of the board of a co-operative society is twenty-one members.,correct,yes,"The model answer correctly states the maximum number of directors as twenty-one, which is supported by the retrieved chunks.",none
8,coi-221: Is a merger of political parties treated as defection under the Tenth Schedule?,A merger of political parties is not treated as defection under the Tenth Schedule.,partly_correct,yes,The model answer correctly states that a merger is not treated as defection but does not mention the specific exemption under Paragraph 4 of the Tenth Schedule.,None
9,coi-248: A 27-year-old citizen wants to contest elections to both the Lok Sabha and the Rajya Sabha. Are they eligible for both?,A 27-year-old citizen is eligible to contest elections to both the Lok Sabha and the Rajya Sabha.,wrong,no,"The model answer incorrectly states that there is no minimum age specified for the Lok Sabha that exceeds the eligibility for the Rajya Sabha, while the correct answer specifies different age requirements for each.","""Yes, a 27-year-old citizen is eligible to contest elections to both the Lok Sabha and the Rajya Sabha as there is no minimum age specified for the Lok Sabha that exceeds the eligibility for the Rajya Sabha."""


,Count out of 10
Correct,6
Partly correct,1
Wrong,3
Supported by the chunks,9


### D2.2 — Spot-check the judge

We read three answers and their chunks ourselves: one the judge marked correct, plus the first two it did not mark *correct + supported*. The full material is printed below.

In [48]:
ok = [i for i, v in enumerate(verdicts) if v["correct"] == "correct" and v["supported"] == "yes"]
bad = [i for i, v in enumerate(verdicts) if i not in ok]
SPOT = (ok[:1] + bad[:2] + ok[1:] + bad[2:])[:3]
for i in SPOT:
    qq, v = queries[i], verdicts[i]
    print("═" * 110 + f"\n{qq['id']}: {qq['question']}\nEXPECTED: {qq['expected']}\n")
    for n, (cid, t) in enumerate(zip(ctx_ids[i], ctx_texts[i]), 1):
        print(f"[{n}] (chunk #{cid}, {chunks[cid]['label']}, contains answer span: {is_hit(t, qq['span'])})\n{t}\n")
    print(f"ANSWER: {answers[i]}\n\nJUDGE → CORRECT: {v['correct']} | SUPPORTED: {v['supported']}\n        {v['reason']}\n")

SPOT_IN_TEXT = {"coi-011": ("correct", "yes"), "coi-067": ("wrong", "yes"), "coi-129": ("wrong", "yes")}
seen = {queries[i]["id"]: (verdicts[i]["correct"], verdicts[i]["supported"]) for i in SPOT}
print("✓ the verdicts discussed in D2.2 match this run" if seen == SPOT_IN_TEXT else
      f"⚠ D2.2 discusses {SPOT_IN_TEXT}, but this run gave {seen}: please update the text")

══════════════════════════════════════════════════════════════════════════════════════════════════════════════
coi-011: On which grounds does the Constitution forbid the State from discriminating against citizens under Article 15?
EXPECTED: Article 15 bars the State from discriminating against any citizen only on grounds of religion, race, caste, sex or place of birth. It also forbids restrictions on access to shops, restaurants, wells and public places on those grounds, while still allowing special provisions for women, children and backward classes.

[1] (chunk #5, Art. 13 … Art. 15, contains answer span: True)
or abridges the rights conferred by this Part and any law made in contravention of this clause shall, to the extent of the contravention, be void.
(3) In this article, unless the context otherwise requires,—
(a) “law” includes any Ordinance, order, bye-law, rule, regulation, notification, custom or usage having in the territory of India the force of law;
(b) “laws in force” in

**Our reading of the three verdicts.** We agree with **all 3 verdicts**, but on one of them (coi-067) we disagree with the judge's *reason*.

| Query | Judge | Our reading | Agree? |
|---|---|---|---|
| coi-011: grounds in Article 15 | correct, supported | The answer gives exactly the grounds of Art. 15(1) (religion, race, caste, sex, place of birth), and chunk [1] states them word for word. It leaves out clause (2) (access to shops, wells …) and the special provisions mentioned in the expected answer. The question asks only for the grounds, though, so *correct* is fair. One flaw that neither check covers: the answer cites **[2]**, which is Article 16 with a different list (it adds "descent" and "residence"). The right citation is [1]. | Yes |
| coi-067: maximum strength of the House of the People | wrong, supported | Our marking scheme counts "Not found in the documents" as wrong. A reply with no claim is supported by definition. So both labels follow the scheme. | Verdict yes, **reason no** |
| coi-129: share of Panchayat seats reserved for women | wrong, supported | The answer is in the chunks, so "Not found" is a real miss. Article 243D(3) starts at the end of chunk [2] ("Not less than one-third (…) of the total") and continues at the top of chunk [1] ("number of seats to be filled by direct election in every Panchayat shall be reserved for women"). Chunk [1] also holds, complete, the one-third rule for women chairpersons in 243D(4). | Yes |

**The disagreement (coi-067).** The judge's reason says "the information is available in the documents". It is not. None of the three chunks holds the 530 + 20 member cap: that text is in chunk #40 (Article 81(1)), which reranking pushed out of the top 3 (see D2.3). Given the chunks it received, "Not found in the documents" was the *right* reply from the generator, and the judge blamed generation for a retrieval failure. The judge most likely saw the cap in `<expected_answer>` and assumed the chunks contained it too. Our two checks cannot tell "refused correctly, because retrieval missed" apart from "refused wrongly". A third check, *ANSWERABLE: do the chunks contain the key facts?*, would separate the two cases.

**Beyond the three.** Reading all ten full answers shows two more weak spots:
* **Wrong citations get through.** coi-094 cites **[4]** although only three chunks were given. The model mixed up clause (4) of Article 124 with a chunk number. Our judge prompt says to ignore chunk numbers, so neither check catches a citation that is wrong or invalid.
* **The judge is lenient on references.** coi-221 says the rules are in "Article 102(2)(1)(a) and (b)". They are in paragraph 2(1)(a)–(b) of the Tenth Schedule, and Article 102 is not in its chunks at all, but the judge still marked the answer *supported*. We would mark it *not supported*.

### D2.3 — Find the cause of one failure

We take one answer the judge marked wrong, partly correct or unsupported, and show its three chunks. **Was the needed information in the chunks?** If the answer span is in one of the three chunks, retrieval did its job and the failure came from **generation**. If not, the failure came from **retrieval**.

In [49]:
fail = next((i for i, v in enumerate(verdicts) if v["correct"] != "correct" or v["supported"] == "no"), None)
if fail is None:
    print("The judge passed every answer, so no failure to diagnose.")
else:
    qq = queries[fail]
    print(f"{qq['id']}: {qq['question']}\nEXPECTED: {qq['expected']}\nANSWER:   {answers[fail]}\n"
          f"JUDGE:    correct={verdicts[fail]['correct']}, supported={verdicts[fail]['supported']} — {verdicts[fail]['reason']}\n")
    for n, (cid, t) in enumerate(zip(ctx_ids[fail], ctx_texts[fail]), 1):
        print(f"[{n}] chunk #{cid} ({chunks[cid]['label']}) — contains answer span: {is_hit(t, qq['span'])}\n{t}\n")
    in_chunks = any(is_hit(t, qq["span"]) for t in ctx_texts[fail])
    print("DIAGNOSIS:", "the answer span IS in the chunks → generation problem" if in_chunks
          else "the answer span is NOT in the chunks → retrieval problem")
    order10, scores10 = rerank(qq["question"], before10[fail], keep=10)
    gold = [c for c in before10[fail] if is_hit(chunks[c]["text"], qq["span"])]
    for c in gold:
        print(f"chunk #{c} ({chunks[c]['label']}) holds the answer span: rank {before10[fail].index(c) + 1} in the dense top 10, "
              f"rank {order10.index(c) + 1} after reranking (score {scores10[order10.index(c)]:.2f}; "
              f"cross-encoder scores of its top 10 range from {min(scores10):.2f} to {max(scores10):.2f})")
    if not gold:
        print("no chunk in the dense top 10 holds the answer span")
    print("✓ D2.3 below matches this run" if qq["id"] == "coi-067" and not in_chunks else
          f"⚠ D2.3 discusses coi-067 (retrieval), but this run picked {qq['id']} ({'generation' if in_chunks else 'retrieval'}): please update the text")

coi-067: What is the maximum strength of the House of the People?
EXPECTED: Article 81 caps the House of the People at not more than five hundred and thirty members directly elected from the States and not more than twenty members representing the Union territories. Seats are allotted so that the population-to-seat ratio is, as far as practicable, the same for all States.
ANSWER:   Not found in the documents
JUDGE:    correct=wrong, supported=yes — The model answer makes no factual statement, which is incorrect as the information is available in the documents.

[1] chunk #41 (Art. 81 … Art. 82) — contains answer span: False
the House of the People in such manner that the ratio between that number and the population of the State is, so far as practicable, the same for all States; and
(b) each State shall be divided into territorial constituencies in such manner that the ratio between the population of each constituency and the number of seats allotted to it is, so far as practicable, th

**Cause: retrieval, and specifically the reranking step (coi-067).** The cell above shows the three chunks the generator received. None of them states the cap:
* [1] (#41) is the second half of Article 81, on allotting seats by population, plus Article 82.
* [2] (#216) is about seats reserved for SC/ST and women in the House of the People (Articles 330–330A).
* [3] (#221) covers Articles 334A–336.

The sentence the question needs, *"not more than five hundred and thirty members … and not more than twenty members"*, is in chunk **#40** (Articles 79–81). Dense search ranked #40 **2nd**, so before reranking it was in the context. The question's words "maximum" and "strength" never appear in the legal text. The cross-encoder gave all ten candidates low and almost equal scores (printed above), and #40 landed **4th**, just past the top-3 cut-off. Chunk #41 repeats "House of the People" many times, and it won. The needed information was **not** in the chunks, so the generator was right to say "Not found". The failure is in retrieval, not generation.

For contrast, **coi-129** (see D2.2) is a *generation* failure. Its answer *is* in the chunks, but the one-third clause is split across the boundary between chunks #135 and #136, and the two pieces reach the model in reverse order ([2] holds the first half, [1] the second). The model did not join them. The same model answered correctly whenever the fact sat in one piece.

### D2.4 — Suggested fix

**Keep the reranker, but send the top 5 reranked chunks to the generator instead of the top 3.**
* **It fixes coi-067.** The cross-encoder's scores for this query are so close that the cut-off at 3 is close to arbitrary. #40 is 4th, so with 5 chunks it is back in the context.
* **It costs little.** Over all 10 queries, the share whose answer span reaches the generator goes back up from 80% to 90%, the same as the dense top 3, and we keep reranking's better hit rate@1. The context grows from about 1,200 to about 2,000 tokens. That is still far inside Qwen2.5-7B-Instruct's 32K window, and it adds only a little prompt-processing time.
* **Why not the alternatives?** Dropping the reranker also brings #40 back, but loses the hit@1 gain from Part C. Overlapping chunks would help coi-129 and coi-248, whose answers are split across chunk boundaries, but would not help coi-067, whose answer chunk was retrieved and then cut.

The cell below tests the fix. It measures the retrieval side for all 10 queries, then re-generates the failed answer with the top 5 chunks.

In [50]:
FIX_KEEP = 5
fix_ids = [rerank(q, b, keep=FIX_KEEP)[0] for q, b in zip(Q, before10)]
fix_hit = 100 * sum(any(is_hit(chunks[c]["text"], qq["span"]) for c in ids) for ids, qq in zip(fix_ids, queries)) / len(Q)
print(f"queries whose answer span reaches the generator: top 3 after reranking {hit3_after:.0f}% → top {FIX_KEEP} after reranking "
      f"{fix_hit:.0f}%  (context ≈ {3 * CHOSEN_CHUNK_SIZE} → {FIX_KEEP * CHOSEN_CHUNK_SIZE} tokens)")
fix_result = None
if fail is not None:
    qq, texts = queries[fail], [chunks[c]["text"] for c in fix_ids[fail]]
    print(f"{qq['id']} now gets: {show_ids(fix_ids[fail], qq['span'])}")
    if SWITCH:
        fix_tok, fix_kw = gen_tok, {}
        fix_model = load_causal_lm(GENERATOR).eval()
        prompt = chat_prompt(gen_tok, INSTRUCT_SYSTEM, instruct_user(qq["question"], texts))
    else:
        fix_tok, fix_kw = a1_tok, dict(repetition_penalty=1.1, eos_token_id=a1_tok.eos_token_id)
        fix_model = PeftModel.from_pretrained(load_causal_lm(A1_INFO["cpt_merged_dir"]), A1_INFO["sft_adapter_dir"]).eval()
        prompt = a1_prompt(qq["question"], texts)
    new_answer = re.split(r"\n\s*###", generate(fix_model, fix_tok, [prompt], **fix_kw)[0])[0].strip()
    del fix_model; free_memory()
    fix_result = dict(id=qq["id"], keep=FIX_KEEP, chunk_ids=fix_ids[fail], answer_top3=answers[fail], answer_fixed=new_answer)
    print(f"\nEXPECTED:            {qq['expected']}\nANSWER with top 3:   {answers[fail]}\nANSWER with top {FIX_KEEP}:   {new_answer}")
    print(f"\nanswer chunk now in the context: {any(is_hit(t, qq['span']) for t in texts)} | "
          f"generator still says 'Not found': {says_not_found(new_answer)}")

queries whose answer span reaches the generator: top 3 after reranking 80% → top 5 after reranking 90%  (context ≈ 1200 → 2000 tokens)
coi-067 now gets: #41 Art. 81 … Art. 82 | #216 Art. 330 … Art. 330A | #221 Art. 334A … Art. 336 | #40 Art. 79 … Art. 81 ✓ | #238 Art. 352


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]


EXPECTED:            Article 81 caps the House of the People at not more than five hundred and thirty members directly elected from the States and not more than twenty members representing the Union territories. Seats are allotted so that the population-to-seat ratio is, as far as practicable, the same for all States.
ANSWER with top 3:   Not found in the documents
ANSWER with top 5:   The House of the People shall consist of not more than five hundred and thirty members chosen by direct election from territorial constituencies in the States [4].

answer chunk now in the context: True | generator still says 'Not found': False


**Result of the fix.** With the top 5 reranked chunks, #40 reaches the generator as chunk [4]. The answer changes from "Not found in the documents" to *"The House of the People shall consist of not more than five hundred and thirty members chosen by direct election from territorial constituencies in the States [4]"*. That is grounded and correctly cited.
* **Retrieval:** the share of queries whose answer span reaches the generator goes from 80% to 90%.
* **Still incomplete:** the answer leaves out clause (b), "not more than twenty members" from the Union territories, which is in the same chunk. Under our marking scheme it would now count as *partly correct* rather than *wrong*. Adding "include every number or limit the chunks state" to the prompt would address the remaining gap.

## Save results

In [51]:
out = {
    "chunk_size_table": a2_table.reset_index().to_dict("records"),
    "chosen_chunk_size": CHOSEN_CHUNK_SIZE,
    "dense_index": {k: (str(v) if not isinstance(v, (int, float)) else v) for k, v in dense_stats[CHOSEN_CHUNK_SIZE].items()},
    "search_table": b_table.reset_index().to_dict("records"),
    "best_method": BEST_METHOD,
    "rerank": dict(ms_per_query=float(np.mean(rr_ms)), rank_change_rate=rank_change, hit1_before=hit1_before,
                   hit1_after=hit1_after, hit3_before=hit3_before, hit3_after=hit3_after),
    "generator": GENERATOR, "judge": JUDGE,
    "answers": [dict(question=q, chunk_ids=ids, answer=a, a1_answer=a1)
                for q, ids, a, a1 in zip(all_questions, ctx_ids, answers, a1_answers)],
    "verdicts": [dict(id=qq["id"], **v, raw=r) for qq, v, r in zip(queries, verdicts, judge_raw)],
    "fix_test": dict(keep=FIX_KEEP, hit_rate_in_context=fix_hit, failed_answer=fix_result),
}
(RESULTS_DIR / "rag_results.json").write_text(json.dumps(out, indent=1, ensure_ascii=False, default=str))
judge_table.to_csv(RESULTS_DIR / "judge_table.csv", index=False)
print("saved to", RESULTS_DIR)

saved to /content/drive/MyDrive/llm_assignment_2/results/rag


### Export to HTML (for submission)
Save the notebook first (**File → Save**), then run the cell below. It writes `rag_pipeline.html` next to the results on Drive. Change `NOTEBOOK_PATH` if you saved the notebook somewhere else.

In [52]:
NOTEBOOK_PATH = Path("/content/drive/MyDrive/Colab Notebooks/rag_pipeline.ipynb")
if IN_COLAB and NOTEBOOK_PATH.exists():
    subprocess.run(["jupyter", "nbconvert", "--to", "html", str(NOTEBOOK_PATH), "--output-dir", str(RESULTS_DIR)], check=True)
    print("HTML written to", RESULTS_DIR)
else:
    print(f"Notebook not found at {NOTEBOOK_PATH}: set NOTEBOOK_PATH (or use File → Download → .html).")

HTML written to /content/drive/MyDrive/llm_assignment_2/results/rag
